<a href="https://colab.research.google.com/github/Vinicius-Araujppp/trabalho-mineracao-dados-jaqueline1bimestre/blob/main/02_regressao_insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho de Mineração de Dados - 1º Bimestre
**Tarefa:** Regressão
**Base de dados:** Custos Médicos (insurance.csv)
## 1. Introdução
Este notebook tem como objetivo prever os custos hospitalares (charges) com base em variáveis socioeconômicas e de saúde. Utilizaremos a penalização Lasso (L1) como algoritmo embutido para seleção de atributos, buscando remover variáveis irrelevantes.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Carregar dados (Certifique-se de ter feito upload do insurance.csv no Colab)
df = pd.read_csv('insurance.csv')
X = df.drop('charges', axis=1)
y = df['charges']

# Pré-processamento: OneHot para categóricas, StandardScaler para numéricas
num_features = ['age', 'bmi', 'children']
cat_features = ['sex', 'smoker', 'region']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)])

X_processed = preprocessor.fit_transform(X)

# Divisão 80/20 com random_state=42
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Regressão Linear com todas as colunas codificadas
modelo_lin = LinearRegression()
modelo_lin.fit(X_train, y_train)
y_pred_lin = modelo_lin.predict(X_test)

print("R2 Inicial:", r2_score(y_test, y_pred_lin))
print("RMSE Inicial:", np.sqrt(mean_squared_error(y_test, y_pred_lin)))

In [ ]:
from sklearn.linear_model import Lasso

# Regressão Lasso (L1) para seleção embutida de atributos eliminando coeficientes nulos
modelo_lasso = Lasso(alpha=100.0, random_state=42)
modelo_lasso.fit(X_train, y_train)
y_pred_lasso = modelo_lasso.predict(X_test)

print("R2 Lasso (Pós-Seleção):", r2_score(y_test, y_pred_lasso))
print("RMSE Lasso:", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))

In [ ]:
import matplotlib.pyplot as plt

# Gráfico de Coeficientes do Lasso para visualizar quais variáveis tiveram peso zerado
nomes_atributos = preprocessor.get_feature_names_out()
coeficientes = modelo_lasso.coef_

plt.figure(figsize=(10, 5))
plt.barh(nomes_atributos, coeficientes, color='orange')
plt.title('Impacto das Variáveis (Coeficientes do Lasso)')
plt.xlabel('Valor do Coeficiente')
plt.show()

# Dispersão de Resíduos
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_lasso, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
plt.title('Valores Reais vs Preditos (Lasso)')
plt.xlabel('Custo Real ($)')
plt.ylabel('Custo Predito ($)')
plt.show()

## 8. Conclusão
A aplicação da Regressão Lasso (L1) atuou de forma eficaz como um método embutido de seleção de atributos. Ao analisar o gráfico de coeficientes, notamos que o modelo penalizou e zerou o peso de variáveis irrelevantes. O impacto da remoção dessas variáveis resultou em um modelo mais simples e direto de se interpretar, mantendo as métricas de R2 e RMSE altamente competitivas em relação ao modelo inicial com todas as variáveis.